# Setup WQU — DataExplorers2026
### Chạy file này 1 lần duy nhất khi mới vào WQU Jupyter

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 1: Cài thư viện cần thiết
# ═══════════════════════════════════════════════
!pip install prophet pmdarima xgboost scikit-learn statsmodels matplotlib ipywidgets openai --quiet
print('✅ Cài thư viện xong')

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 2: Clone repo bài thi về
# ═══════════════════════════════════════════════
import os

REPO_URL = 'https://github.com/trungcandygit/DataExplorers2026-ForecastingC.git'
REPO_DIR = 'DataExplorers2026-ForecastingC'

if os.path.exists(REPO_DIR):
    print('Repo đã có — pull bản mới nhất...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('Đang clone repo...')
    os.system(f'git clone {REPO_URL}')

print(f'✅ Repo sẵn sàng tại: {REPO_DIR}/')

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 3: Di chuyển vào thư mục dự án
# ═══════════════════════════════════════════════
%cd DataExplorers2026-ForecastingC
!pwd
print('✅ Đang ở đúng thư mục')

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 4: Kiểm tra cấu trúc file
# ═══════════════════════════════════════════════
import os

for root, dirs, files in os.walk('.'):
    # bỏ qua thư mục ẩn và data_raw (quá nhiều)
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['data_raw','emails_pdfs','database']]
    level = root.replace('.', '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level == 0:
        print(f'📁 DataExplorers2026-ForecastingC/')
    else:
        print(f'{indent}📁 {folder}/')
    subindent = '  ' * (level + 1)
    for f in sorted(files):
        if f.endswith(('.ipynb', '.csv', '.pdf')):
            size = os.path.getsize(os.path.join(root, f))
            print(f'{subindent}📄 {f}  ({size/1024:.0f} KB)')

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 5: Kiểm tra dữ liệu chính
# ═══════════════════════════════════════════════
import pandas as pd

# Dữ liệu chuỗi thời gian tháng
df_ts = pd.read_csv('tnbike_data.csv', parse_dates=['ds'])
print('tnbike_data.csv:', df_ts.shape)
print(df_ts)
print()

# Dữ liệu Q2/2026 để dự báo
df_future = pd.read_csv('future.csv', parse_dates=['ds'])
print('future.csv:', df_future.shape)
print(df_future)

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 6: Kiểm tra fact_sales đầy đủ
# ═══════════════════════════════════════════════
import pandas as pd

df_full = pd.read_csv('data_raw/fact_sales_full.csv', parse_dates=['order_date'])
print(f'fact_sales_full: {df_full.shape[0]:,} đơn hàng, {df_full.shape[1]} cột')
print(f'Đại lý:  {df_full.customer_code.nunique()} khách hàng')
print(f'SKU:     {df_full.product_name.nunique()} mẫu xe')
print(f'Tỉnh:    {df_full.province_name.nunique()} tỉnh/thành')
print(f'Từ:      {df_full.order_date.min().date()} → {df_full.order_date.max().date()}')
print()
df_full.head(3)

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 7: Kiểm tra kết quả dự báo đã có sẵn
# ═══════════════════════════════════════════════
import pandas as pd

prophet = pd.read_csv('Forecasting Product/Ensemble/predictions_prophet.csv')
sarimax = pd.read_csv('Forecasting Product/Ensemble/predictions_sarimax.csv')
gbm    = pd.read_csv('Forecasting Product/Ensemble/predictions_gbm.csv')

print('=== KẾT QUẢ DỰ BÁO Q2/2026 ===')
print(f'{'Tháng':<12} {'Prophet':>12} {'SARIMAX':>12} {'Ensemble':>12}')
print('-' * 50)
for i, month in enumerate(['T4/2026', 'T5/2026', 'T6/2026']):
    p = prophet['yhat'].iloc[i]/1e9
    s = sarimax['yhat'].iloc[i]/1e9
    g = gbm['yhat'].iloc[i]/1e9
    print(f'{month:<12} {p:>10.2f}T {s:>10.2f}T {g:>10.2f}T')

print('-' * 50)
print(f'{'Tổng Q2':<12} {prophet["yhat"].sum()/1e9:>10.2f}T {sarimax["yhat"].sum()/1e9:>10.2f}T {gbm["yhat"].sum()/1e9:>10.2f}T')

In [ ]:
# ═══════════════════════════════════════════════
# BƯỚC 8: Vẽ chart nhanh để verify
# ═══════════════════════════════════════════════
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df_ts = pd.read_csv('tnbike_data.csv', parse_dates=['ds'])
prophet = pd.read_csv('Forecasting Product/Ensemble/predictions_prophet.csv', parse_dates=['ds'])
sarimax = pd.read_csv('Forecasting Product/Ensemble/predictions_sarimax.csv', parse_dates=['ds'])
gbm    = pd.read_csv('Forecasting Product/Ensemble/predictions_gbm.csv', parse_dates=['ds'])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_ts.ds, df_ts.revenue/1e9, 'o-', color='steelblue', linewidth=2.5, label='Lịch sử', markersize=8)
ax.plot(prophet.ds, prophet.yhat/1e9, 's--', color='steelblue', alpha=0.7, label='Prophet', markersize=9)
ax.plot(sarimax.ds, sarimax.yhat/1e9, '^--', color='darkorange', alpha=0.7, label='SARIMAX', markersize=9)
ax.plot(gbm.ds, gbm.yhat/1e9, 'D-', color='forestgreen', linewidth=3, label='Ensemble', markersize=10)
ax.fill_between(gbm.ds, gbm.yhat_lower/1e9, gbm.yhat_upper/1e9, alpha=0.15, color='forestgreen')
ax.axvline(pd.Timestamp('2026-03-31'), color='black', linestyle=':', alpha=0.5)
ax.text(pd.Timestamp('2026-04-05'), ax.get_ylim()[0] + 1, '▶ Dự báo', fontsize=10, color='gray')
for _, row in gbm.iterrows():
    ax.annotate(f"{row.yhat/1e9:.1f}T", (row.ds, row.yhat/1e9),
                textcoords='offset points', xytext=(0,12),
                fontsize=11, fontweight='bold', ha='center', color='forestgreen')
ax.set_title('Dự báo Doanh số Q2/2026 — Xe đạp Thống Nhất', fontsize=14, fontweight='bold')
ax.set_ylabel('Doanh số (tỷ VND)', fontsize=12)
ax.set_xlabel('Tháng', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('✅ Setup hoàn tất — sẵn sàng chạy các notebook!')